In [ ]:
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.static import teams
import pandas as pd
import time

pd.set_option('display.max_columns', None)

nba_teams = teams.get_teams()
all_team_logs = []

print("Fetching all 2024-25 regular season games by team...")


try:
    # Get ALL games for the 2024-25 season at once
    gamefinder = leaguegamefinder.LeagueGameFinder(
        season_nullable='2024-25',
        season_type_nullable='Regular Season'
    )
    all_games_df = gamefinder.get_data_frames()[0]

    # Loop through team list to group and label
    for team in nba_teams:
        team_id = team['id']
        team_name = team['full_name']
        print(f"Processing: {team_name}")

        team_df = all_games_df[all_games_df['TEAM_ID'] == team_id].copy()
        team_df['TEAM_NAME'] = team_name

        if not team_df.empty:
            all_team_logs.append(team_df)
        else:
            print(f" No data found for {team_name}")

        time.sleep(0.5)

    # Combine all team data
    season_df = pd.concat(all_team_logs, ignore_index=True)
    print(f"\nTotal rows collected: {len(season_df)}")
    display(season_df.head())

except Exception as e:
    print(f"Error fetching data: {e}")



In [ ]:
df = season_df
df = df.sort_values(by='GAME_DATE')

# Rolling averages (3-game window)
df['PTS_rolling_avg'] = df.groupby('TEAM_NAME')['PTS'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['AST_rolling_avg'] = df.groupby('TEAM_NAME')['AST'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['REB_rolling_avg'] = df.groupby('TEAM_NAME')['REB'].transform(lambda x: x.rolling(3, min_periods=1).mean())

# Win/Loss to binary
df['WIN'] = df['WL'].map({'W': 1, 'L': 0})

# Win streak
df['WIN_STREAK'] = df.groupby('TEAM_NAME')['WIN'].transform(lambda x: x * (x.groupby((x != x.shift()).cumsum()).cumcount() + 1))

# Home game flag
df['HOME_GAME'] = df['MATCHUP'].str.contains("vs.").astype(int)

display(df.head())

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# extract opponent abbreviation from MATCHUP
opp_abbr = df['MATCHUP'].str.extract(r'(?:vs\.|@)\s+([A-Z]{2,3})')[0]

# build mapping from abbreviation -> full_name using nba_teams (defined earlier)
full_by_abbr = {t['abbreviation']: t['full_name'] for t in nba_teams}

# map abbreviation to full team name so it matches df['TEAM_NAME']
df['OPP_TEAM_NAME'] = opp_abbr.map(full_by_abbr)

# drop rows where mapping failed
df = df.dropna(subset=['OPP_TEAM_NAME', 'TEAM_NAME']).copy()

# Aggregate team stats
team_stats = df.groupby('TEAM_NAME')[[
    'PTS_rolling_avg', 'AST_rolling_avg', 'REB_rolling_avg', 'WIN_STREAK'
]].mean().reset_index()

# Build matchup dataset
matchup_data = []
for idx, row in df.iterrows():
    team = row['TEAM_NAME']
    opp = row['OPP_TEAM_NAME']

    # skip if either team not found in team_stats
    if team not in team_stats['TEAM_NAME'].values or opp not in team_stats['TEAM_NAME'].values:
        continue

    home_stats = team_stats[team_stats['TEAM_NAME'] == team]
    away_stats = team_stats[team_stats['TEAM_NAME'] == opp]

    matchup_data.append({
        'HOME_PTS_avg': home_stats['PTS_rolling_avg'].values[0],
        'HOME_AST_avg': home_stats['AST_rolling_avg'].values[0],
        'HOME_REB_avg': home_stats['REB_rolling_avg'].values[0],
        'HOME_WIN_STREAK': home_stats['WIN_STREAK'].values[0],
        'AWAY_PTS_avg': away_stats['PTS_rolling_avg'].values[0],
        'AWAY_AST_avg': away_stats['AST_rolling_avg'].values[0],
        'AWAY_REB_avg': away_stats['REB_rolling_avg'].values[0],
        'AWAY_WIN_STREAK': away_stats['WIN_STREAK'].values[0],
        'HOME_GAME': row['HOME_GAME'],
        'WIN': row['WIN']
    })

matchup_df = pd.DataFrame(matchup_data)

# Train/test split
features = [
    'HOME_PTS_avg', 'HOME_AST_avg', 'HOME_REB_avg', 'HOME_WIN_STREAK',
    'AWAY_PTS_avg', 'AWAY_AST_avg', 'AWAY_REB_avg', 'AWAY_WIN_STREAK',
    'HOME_GAME'
]
X = matchup_df[features]
y = matchup_df['WIN']
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Train model
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

# Build prediction matrix
team_names = team_stats['TEAM_NAME'].tolist()
matrix = pd.DataFrame(index=team_names, columns=team_names)

for home in team_names:
    for away in team_names:
        if home == away:
            matrix.loc[home, away] = '-'
        else:
            home_row = team_stats[team_stats['TEAM_NAME'] == home]
            away_row = team_stats[team_stats['TEAM_NAME'] == away]

            input_features = pd.DataFrame([{
                'HOME_PTS_avg': home_row['PTS_rolling_avg'].values[0],
                'HOME_AST_avg': home_row['AST_rolling_avg'].values[0],
                'HOME_REB_avg': home_row['REB_rolling_avg'].values[0],
                'HOME_WIN_STREAK': home_row['WIN_STREAK'].values[0],
                'AWAY_PTS_avg': away_row['PTS_rolling_avg'].values[0],
                'AWAY_AST_avg': away_row['AST_rolling_avg'].values[0],
                'AWAY_REB_avg': away_row['REB_rolling_avg'].values[0],
                'AWAY_WIN_STREAK': away_row['WIN_STREAK'].values[0],
                'HOME_GAME': 1
            }])

            prob = model.predict_proba(input_features)[0][1]
            matrix.loc[home, away] = prob

display(matrix)

# Row = Home team
# Column = Away team
# Value = Probability of the home team winning

In [ ]:
import plotly.express as px
import numpy as np
import plotly.io as pio
pio.renderers.default = "browser"

# Clean the matrix for heatmap
plot_matrix = matrix.replace('-', np.nan).replace('%', '',regex=True).astype(float)

# Plot
fig = px.imshow(
    plot_matrix,
    labels=dict(x="Away Team", y="Home Team", color="Win Probability"),
    x=plot_matrix.columns,
    y=plot_matrix.index,
    color_continuous_scale='RdYlGn',
    zmin=0, zmax=1,
    width=1100, height=1000
)
fig.update_layout(
    title="NBA Predianator",
    xaxis_title="Away Team",
    yaxis_title="Home Team"
)

fig.show()